In [ ]:
EXP_NAME = "0615_uci_SPS"
dataset = "heart_disease_train"
feature_map = "sps/features"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from src.config import Config

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{dataset}.csv')

with open(f"{PROJECT_ROOT}/configs/{feature_map}.json", 'r') as f:
  features = json.load(f)

# Analysis setup

In [ ]:
full_audit = pd.concat([sps_audit, sbs_audit_baseline], ignore_index=True, axis=0)
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['iteration'])['feature'].apply(set).to_dict()

iteration_per_feature = full_audit[full_audit['bucket'] == 'x_desc'].groupby('feature')[['iteration']].count()
iteration_per_feature

In [ ]:
# full_audit.describe().T

In [ ]:


# CHANCE LEVEL AUPRC FOR REFERENCE
positives = dataset[dataset[features["target"]["name"]] == 1]
auprc_chance = len(positives) / len(dataset)
subgroups = dataset[features["sens"][0]["name"]].unique().astype(int)

# UTILITY VALUE
full_audit['auprc_value'] = full_audit['u_global_mean_auprc'] - full_audit['abla_global_mean_auprc']
full_audit['min_auprc_value'] = 1
for g in subgroups:
  full_audit[f'{g}_auprc_value'] = full_audit[f'u_{g}_mean_auprc'] - full_audit[f'abla_{g}_mean_auprc']
  full_audit['min_auprc_value'] = full_audit[[f'{g}_auprc_value', 'min_auprc_value']].min(axis=1)

# UTILITY COST
full_audit['auprc_cost'] = full_audit['x_global_mean_auprc'] - full_audit['u_global_mean_auprc']

full_audit['max_auprc_cost'] = -1
for g in subgroups:
  full_audit[f'{g}_auprc_cost'] = full_audit[f'x_{g}_mean_auprc'] - full_audit[f'u_{g}_mean_auprc'] 
  full_audit['max_auprc_cost'] = full_audit[[f'{g}_auprc_cost', 'max_auprc_cost']].max(axis=1)

# CF HARM REDUCTION
full_audit['max_cf_harm_reduction'] = 0
for g in subgroups:
  full_audit['max_cf_harm_reduction'] = full_audit[[f'x_{g}_bal_harm_mean', 'max_cf_harm_reduction']].max(axis=1)



# Config trade-off analysis

In [ ]:
tradeoff_audit = full_audit.groupby('iteration')[['max_auprc_cost', 'max_cf_harm_reduction', 'x_global_mean_auprc', 'abla_global_mean_auprc', 'min_auprc_value']].median().round(4)

In [ ]:
# Raw features AUPRC = Max group AUPRC of the X probe when all features are in Xdesc
full_x_desc_auprc = tradeoff_audit.loc[-1, "x_global_mean_auprc"]

# Raw features CF harm = Max group CF harm of the X probe when all features are in Xdesc
full_x_desc_cf_harm = tradeoff_audit.loc[-1, "max_cf_harm_reduction"]

In [ ]:
utility_vs_fairness = tradeoff_audit.sort_values('max_auprc_cost')

pareto_frontier = []
max_max_cf_harm_reduction = 0

print('\n--- Configurations on the AUPRC Cost / S-info Loss Pareto Frontier ---')

for idx, solution in utility_vs_fairness.iterrows():
  if solution['max_cf_harm_reduction'] > max_max_cf_harm_reduction:
    # Normalised CF harm reduction: 
    # = CF harm reduction / Raw features CF harm
    solution['norm_cf_harm_reduction'] = solution['max_cf_harm_reduction'] / full_x_desc_cf_harm
    # Normalised AUPRC cost:
    # = AUPRC cost / Raw features AUPRC
    solution['norm_auprc_cost'] = solution['max_auprc_cost'] / full_x_desc_auprc

    solution['FEU'] = (solution['norm_auprc_cost'] / solution['norm_cf_harm_reduction']).round(3) if solution['norm_cf_harm_reduction'] else "NA"
    pareto_frontier.append(solution)
    max_max_cf_harm_reduction = solution['max_cf_harm_reduction']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print('\n')
print(pareto_frontier_df.drop('x_global_mean_auprc', axis=1).to_markdown())
print('\nINTERPRETATION:')
print('- FEU > 0 and CF harm reduc. > 0 => The lowest the value, the better trade-off (= high CF harm reduction vs low AUPRC cost)')
print('- FEU < 0 and CF harm reduc. > 0 => Free lunch (=improvement in AUPRC as well as reduction in CF harm)')
print('- CF harm reduc. < 0 => Increase in CF harm')


In [ ]:

best_candidate = pareto_frontier_df[pareto_frontier_df['max_cf_harm_reduction'] > 0 ]
best_candidate = pareto_frontier_df[pareto_frontier_df['FEU'] == pareto_frontier_df['FEU'].min()]
print(f"Config with best utility loss / fairness gain trade-off: {[x_desc_configs.get(idx, "empty") for idx in best_candidate.index.values]}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=pareto_frontier_df, x='max_auprc_cost', y='max_cf_harm_reduction', color="green", marker='', linestyle="--", errorbar=None, ax=ax)
sns.scatterplot(data=utility_vs_fairness, x='max_auprc_cost', y='max_cf_harm_reduction', ax=ax)

for idx, row in pareto_frontier_df.iterrows():
      ax.annotate(
        str(idx),                      
        (row['max_auprc_cost'], row['max_cf_harm_reduction']), 
        textcoords="offset points",    
        xytext=(-10, 5),                 
        ha='left',
        va='bottom',  
        fontsize=9, 
        fontweight='bold', 
        color='#a12aa5'
        )  


plt.xlabel('AUPRC Cost')
plt.ylabel('Counterfactual Harm Reduction')
plt.legend(labels=[ 
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit'
                   ],
           loc='upper left', bbox_to_anchor=(0, -0.1), edgecolor="white")

plt.show()

# All configs

In [ ]:
def find_config(row):
  return x_desc_configs.get(row['id'], "empty Xdesc")

all_configs = utility_vs_fairness.sort_values('min_auprc_value', ascending=False).reset_index(names="id")
all_configs['Xdesc config'] = all_configs.apply(find_config, axis=1)

print(all_configs.drop('x_global_mean_auprc', axis=1).to_markdown(index=False))